In [2]:
import pandas as pd
import numpy as np

# Read the main data
df_main = pd.read_csv("combined_calls_CAR.csv")
df_main['Activity Start Timestamp'] = pd.to_datetime(df_main['Activity Start Timestamp'])
df_main = df_main.sort_values(['Contact Session ID', 'Activity Start Timestamp'])

print("Creating comprehensive front desk dataset...")

# ============================================================
# 1. IDENTIFY FRONT DESK SESSIONS AND ADD FLAGS
# ============================================================

# Mark front desk activities
df_main['is_front_desk'] = df_main['Activity Name'].str.contains('FrontDesk', case=False, na=False)

# Get sessions with front desk interactions
fd_sessions_list = df_main[df_main['is_front_desk']]['Contact Session ID'].unique()

# Add session-level flags
df_main['has_front_desk'] = df_main['Contact Session ID'].isin(fd_sessions_list)

# ============================================================
# 2. GET FRONT DESK TYPE FOR EACH SESSION
# ============================================================

# Get the front desk type for each session (first occurrence)
fd_types = df_main[df_main['is_front_desk']].groupby('Contact Session ID')['Activity Name'].first()

# Merge back to main dataframe
df_main = df_main.merge(
    fd_types.rename('session_fd_type').reset_index(), 
    on='Contact Session ID', 
    how='left'
)

# ============================================================
# 3. ADD TIME COMPONENTS FOR FRONT DESK SESSIONS
# ============================================================

# Get first front desk timestamp for each session
fd_timestamps = df_main[df_main['is_front_desk']].groupby('Contact Session ID')['Activity Start Timestamp'].first()

df_main = df_main.merge(
    fd_timestamps.rename('fd_timestamp').reset_index(), 
    on='Contact Session ID', 
    how='left'
)

# Extract time components from FD timestamp
df_main['fd_hour'] = df_main['fd_timestamp'].dt.hour
df_main['fd_day_of_week'] = df_main['fd_timestamp'].dt.day_name()
df_main['fd_day_of_week_num'] = df_main['fd_timestamp'].dt.dayofweek
df_main['fd_month'] = df_main['fd_timestamp'].dt.month_name()
df_main['fd_month_num'] = df_main['fd_timestamp'].dt.month


# When I re-run this I plan to run this in order to avoid double counting the months for the Month of Year time plot
#df_main['fd_year'] = df_main['fd_timestamp'].dt.year
df_main['fd_year_month'] = df_main['fd_timestamp'].dt.to_period('M').astype(str)  # Creates "2024-03" format
df_main['fd_month_year_display'] = df_main['fd_timestamp'].dt.strftime('%B %Y')  # Creates "March 2024" format

print("Added year-month columns for proper monthly aggregation")

# ============================================================
# 4. CALCULATE FRONT DESK DURATION FOR EACH SESSION
# ============================================================

# Calculate session-level metrics
session_metrics = []

for session_id in fd_sessions_list:
    session_df = df_main[df_main['Contact Session ID'] == session_id].sort_values('Activity Start Timestamp')
    
    # Get FD timestamp and last timestamp
    fd_timestamp = session_df[session_df['is_front_desk']].iloc[0]['Activity Start Timestamp']
    last_timestamp = session_df.iloc[-1]['Activity Start Timestamp']
    
    # Calculate duration
    duration_seconds = (last_timestamp - fd_timestamp).total_seconds()
    duration_minutes = duration_seconds / 60
    
    session_metrics.append({
        'Contact Session ID': session_id,
        'fd_duration_seconds': duration_seconds,
        'fd_duration_minutes': duration_minutes,
        'session_start': session_df.iloc[0]['Activity Start Timestamp'],
        'session_end': last_timestamp,
        'total_activities': len(session_df)
    })

df_metrics = pd.DataFrame(session_metrics)

# Merge duration back to main dataframe
df_main = df_main.merge(df_metrics, on='Contact Session ID', how='left')

# ============================================================
# 5. ADD RETURN TO MAIN MENU ANALYSIS
# ============================================================

# Mark main menu EPs
main_menu_ep_names = [
    'Main Number Telephony EP',
    'Farmworker Main Number Telephony EP'
]
df_main['is_main_menu'] = df_main['EP Name'].isin(main_menu_ep_names)

# Create activity sequence
df_main['activity_sequence'] = df_main.groupby('Contact Session ID').cumcount()

# Find first FD position in each session
fd_positions = df_main[df_main['is_front_desk']].groupby('Contact Session ID')['activity_sequence'].first()
df_main = df_main.merge(
    fd_positions.rename('first_fd_position').reset_index(), 
    on='Contact Session ID', 
    how='left'
)

# Mark activities after FD
df_main['after_fd'] = (
    (df_main['first_fd_position'].notna()) & 
    (df_main['activity_sequence'] > df_main['first_fd_position'])
)

# Find sessions that returned to main menu after FD
main_menu_returns = df_main[
    (df_main['after_fd'] == True) & 
    (df_main['is_main_menu'] == True)
]['Contact Session ID'].unique()

df_main['session_returned_to_main'] = df_main['Contact Session ID'].isin(main_menu_returns)

# ============================================================
# 6. ADD SUMMARY STATISTICS
# ============================================================

# Calculate summary stats by FD type
duration_stats = df_metrics.merge(
    df_main[['Contact Session ID', 'session_fd_type']].drop_duplicates(),
    on='Contact Session ID'
).groupby('session_fd_type')['fd_duration_minutes'].agg([
    ('avg_duration', 'mean'),
    ('median_duration', 'median'),
    ('min_duration', 'min'),
    ('max_duration', 'max')
]).round(2).reset_index()

# Merge stats back to main dataframe
df_main = df_main.merge(
    duration_stats.rename(columns={'session_fd_type': 'temp_fd_type'}),
    left_on='session_fd_type',
    right_on='temp_fd_type',
    how='left'
).drop('temp_fd_type', axis=1)

# ============================================================
# 7. CLEAN UP AND EXPORT
# ============================================================

# Fill NaN values for non-FD sessions
fill_columns = [
    'session_fd_type', 'fd_timestamp', 'fd_hour', 'fd_day_of_week', 
    'fd_day_of_week_num', 'fd_month', 'fd_month_num', 'fd_duration_seconds', 
    'fd_duration_minutes', 'session_start', 'session_end', 'total_activities',
    'first_fd_position', 'avg_duration', 'median_duration', 'min_duration', 'max_duration'
]

for col in fill_columns:
    if col in df_main.columns:
        if 'duration' in col or col in ['fd_hour', 'fd_day_of_week_num', 'fd_month_num', 'total_activities', 'first_fd_position']:
            df_main[col] = df_main[col].fillna(0)
        else:
            df_main[col] = df_main[col].fillna('N/A')

# Fill boolean columns
df_main['has_front_desk'] = df_main['has_front_desk'].fillna(False)
df_main['is_front_desk'] = df_main['is_front_desk'].fillna(False)
df_main['is_main_menu'] = df_main['is_main_menu'].fillna(False)
df_main['after_fd'] = df_main['after_fd'].fillna(False)
df_main['session_returned_to_main'] = df_main['session_returned_to_main'].fillna(False)

# Export the comprehensive dataset
df_main.to_csv('comprehensive_front_desk_data.csv', index=False)

print("=" * 60)
print("COMPREHENSIVE DATASET CREATED")
print("=" * 60)
print(f"Total rows: {len(df_main):,}")
print(f"Front desk sessions: {len(fd_sessions_list):,}")
print(f"Sessions returning to main: {len(main_menu_returns):,}")

print("\nNew columns added:")
print("   - has_front_desk (boolean)")
print("   - session_fd_type (FrontDeskTransfer, etc.)")
print("   - fd_timestamp (when FD transfer occurred)")
print("   - fd_hour, fd_day_of_week, fd_month (time components)")
print("   - fd_duration_minutes (time spent on FD call)")
print("   - session_returned_to_main (boolean)")
print("   - avg_duration, median_duration (stats by FD type)")

print(f"\nExported: comprehensive_front_desk_data.csv ({len(df_main):,} rows)")

/var/folders/0k/z0xdft5548x02pfzbdd0x6w40000gn/T/ipykernel_95269/3990267521.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_main = pd.read_csv("combined_calls_CAR.csv")


Creating comprehensive front desk dataset...
COMPREHENSIVE DATASET CREATED
Total rows: 3,328,626
Front desk sessions: 17,893
Sessions returning to main: 15,146

New columns added:
   - has_front_desk (boolean)
   - session_fd_type (FrontDeskTransfer, etc.)
   - fd_timestamp (when FD transfer occurred)
   - fd_hour, fd_day_of_week, fd_month (time components)
   - fd_duration_minutes (time spent on FD call)
   - session_returned_to_main (boolean)
   - avg_duration, median_duration (stats by FD type)

Exported: comprehensive_front_desk_data.csv (3,328,626 rows)


####  **Volume Breakdown of Calls Among 4 FrontDeskTransfer Types**

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

# Read the data
df_main = pd.read_csv("combined_calls_CAR.csv")
df_main['Activity Start Timestamp'] = pd.to_datetime(df_main['Activity Start Timestamp'])
df_main = df_main.sort_values(['Contact Session ID', 'Activity Start Timestamp'])

# Mark front desk activities
# This groups by Contact Session ID and takes the first front desk transfer type encountered in each session. 
df_main['is_front_desk'] = df_main['Activity Name'].str.contains('FrontDesk', case=False, na=False)

# Get sessions with front desk interactions
sessions_with_fd = df_main[df_main['is_front_desk']]['Contact Session ID'].unique()

# Get the front desk type for each session (first occurrence)
fd_types = df_main[df_main['is_front_desk']].groupby('Contact Session ID')['Activity Name'].first()

# Create volume distribution dataframe
volume_distribution = fd_types.value_counts().reset_index()
volume_distribution.columns = ['Front Desk Type', 'Volume']
volume_distribution['Percentage'] = (volume_distribution['Volume'] / volume_distribution['Volume'].sum() * 100).round(1)

# Add percentage label for visualization
volume_distribution['Label'] = volume_distribution.apply(
    lambda x: f"{x['Volume']:,} ({x['Percentage']}%)", axis=1
)

# Sort by volume descending
volume_distribution = volume_distribution.sort_values('Volume', ascending=False)

print("=" * 60)
print("FRONT DESK VOLUME DISTRIBUTION")
print("=" * 60)
print(volume_distribution)

# Export for Power BI
volume_distribution.to_csv('front_desk_volume_distribution.csv', index=False)
print("\n✅ Exported: front_desk_volume_distribution.csv")

/var/folders/0k/z0xdft5548x02pfzbdd0x6w40000gn/T/ipykernel_82398/2204323668.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_main = pd.read_csv("combined_calls_CAR.csv")


FRONT DESK VOLUME DISTRIBUTION
      Front Desk Type  Volume  Percentage          Label
0  FrontDeskTransfer1    7912        44.2  7,912 (44.2%)
1  FrontDeskTransfer2    6895        38.5  6,895 (38.5%)
2   FrontDeskTransfer    2545        14.2  2,545 (14.2%)
3  FrontDeskTransfer3     541         3.0     541 (3.0%)

✅ Exported: front_desk_volume_distribution.csv


#### **Time Trends (Hour of day, day of week, month of year)**

In [19]:
# Get ONLY the front desk transfer rows
front_desk_activities = df_main[
    df_main['Activity Name'].str.contains('FrontDesk', case=False, na=False)
].copy()

# For sessions with multiple FD activities, take the first one
fd_sessions = front_desk_activities.groupby('Contact Session ID').first().reset_index()

# Extract time components from the FD activity timestamp
fd_sessions['Hour_of_Day'] = fd_sessions['Activity Start Timestamp'].dt.hour
fd_sessions['Day_of_Week'] = fd_sessions['Activity Start Timestamp'].dt.day_name()
fd_sessions['Day_of_Week_Num'] = fd_sessions['Activity Start Timestamp'].dt.dayofweek
fd_sessions['Month'] = fd_sessions['Activity Start Timestamp'].dt.month_name()
fd_sessions['Month_Num'] = fd_sessions['Activity Start Timestamp'].dt.month
fd_sessions['Front Desk Type'] = fd_sessions['Activity Name']

# Create aggregated tables for each time unit

# 1. BY HOUR OF DAY
hourly_trend = fd_sessions.groupby(['Hour_of_Day', 'Front Desk Type']).size().reset_index(name='Volume')
hourly_trend['Time_Unit'] = 'Hour of Day'
hourly_trend = hourly_trend.rename(columns={'Hour_of_Day': 'Time_Value'})

# 2. BY DAY OF WEEK
daily_trend = fd_sessions.groupby(['Day_of_Week', 'Day_of_Week_Num', 'Front Desk Type']).size().reset_index(name='Volume')
daily_trend = daily_trend.sort_values('Day_of_Week_Num')
daily_trend['Time_Unit'] = 'Day of Week'
daily_trend = daily_trend.rename(columns={'Day_of_Week': 'Time_Value'})
daily_trend = daily_trend[['Time_Value', 'Front Desk Type', 'Volume', 'Time_Unit']]

# 3. BY MONTH
monthly_trend = fd_sessions.groupby(['Month', 'Month_Num', 'Front Desk Type']).size().reset_index(name='Volume')
monthly_trend = monthly_trend.sort_values('Month_Num')
monthly_trend['Time_Unit'] = 'Month'
monthly_trend = monthly_trend.rename(columns={'Month': 'Time_Value'})
monthly_trend = monthly_trend[['Time_Value', 'Front Desk Type', 'Volume', 'Time_Unit']]

# Combine all trends
all_trends = pd.concat([hourly_trend, daily_trend, monthly_trend], ignore_index=True)

# Export
all_trends.to_csv('front_desk_time_trends.csv', index=False)
hourly_trend.to_csv('front_desk_hourly_trend.csv', index=False)

print("TIME TREND DATA CREATED")
print(f"Hourly trend rows: {len(hourly_trend)}")
print(f"Daily trend rows: {len(daily_trend)}")
print(f"Monthly trend rows: {len(monthly_trend)}")
print("\nExported:")
print("   - front_desk_time_trends.csv (all time units)")
print("   - front_desk_hourly_trend.csv (hour of day only)")

TIME TREND DATA CREATED
Hourly trend rows: 36
Daily trend rows: 20
Monthly trend rows: 48

Exported:
   - front_desk_time_trends.csv (all time units)
   - front_desk_hourly_trend.csv (hour of day only)


#### **Average/Median Time Spent by Caller on the 4 Front Desk Transfer Types**

In [25]:
import pandas as pd
import numpy as np

# Read the data
df_main = pd.read_csv("combined_calls_CAR.csv")
df_main['Activity Start Timestamp'] = pd.to_datetime(df_main['Activity Start Timestamp'])
df_main = df_main.sort_values(['Contact Session ID', 'Activity Start Timestamp'])

# Get front desk sessions and calculate duration
front_desk_durations = []

# Get all sessions that had front desk transfers
fd_sessions = df_main[
    df_main['Activity Name'].str.contains('FrontDesk', case=False, na=False)
]['Contact Session ID'].unique()

print(f"Processing {len(fd_sessions):,} front desk sessions...")

for session_id in fd_sessions:
    session_df = df_main[df_main['Contact Session ID'] == session_id].sort_values('Activity Start Timestamp')
    
    # Find the front desk transfer activity
    fd_row = session_df[session_df['Activity Name'].str.contains('FrontDesk', case=False, na=False)]
    
    if len(fd_row) == 0:
        continue
    
    # Get the first front desk transfer
    fd_timestamp = fd_row.iloc[0]['Activity Start Timestamp']
    fd_type = fd_row.iloc[0]['Activity Name']
    
    # Get the last activity in the session
    last_timestamp = session_df.iloc[-1]['Activity Start Timestamp']
    
    # Calculate duration from front desk transfer to end of call
    duration_seconds = (last_timestamp - fd_timestamp).total_seconds()
    duration_minutes = duration_seconds / 60
    
    front_desk_durations.append({
        'Contact Session ID': session_id,
        'Front Desk Type': fd_type,
        'FD Start Time': fd_timestamp,
        'Call End Time': last_timestamp,
        'Duration Seconds': duration_seconds,
        'Duration Minutes': duration_minutes
    })

# Create dataframe
df_durations = pd.DataFrame(front_desk_durations)

# Calculate statistics by front desk type
duration_stats = df_durations.groupby('Front Desk Type')['Duration Minutes'].agg([
    ('Count', 'count'),
    ('Average_Minutes', 'mean'),
    ('Median_Minutes', 'median'),
    ('Min_Minutes', 'min'),
    ('Max_Minutes', 'max'),
    ('Std_Minutes', 'std')
]).round(2)

duration_stats = duration_stats.reset_index()

# Add formatted labels for display
duration_stats['Average_Label'] = duration_stats['Average_Minutes'].apply(lambda x: f"{x:.1f} min")
duration_stats['Median_Label'] = duration_stats['Median_Minutes'].apply(lambda x: f"{x:.1f} min")

print("\nFRONT DESK CALL DURATION STATISTICS")
print("="*60)
print(duration_stats.to_string(index=False))

# Export for Power BI
duration_stats.to_csv('front_desk_duration_stats.csv', index=False)
df_durations.to_csv('front_desk_detailed_durations.csv', index=False)

print("\nExported:")
print("   - front_desk_duration_stats.csv (summary statistics)")
print("   - front_desk_detailed_durations.csv (all session details)")

# Quick verification - check for outliers
print("\nOUTLIER CHECK:")
print(f"Sessions > 30 minutes: {len(df_durations[df_durations['Duration Minutes'] > 30])}")
print(f"Sessions > 60 minutes: {len(df_durations[df_durations['Duration Minutes'] > 60])}")

# Show sample of very long calls to verify logic
long_calls = df_durations[df_durations['Duration Minutes'] > 30].head(3)
if len(long_calls) > 0:
    print("\nSample long calls (for verification):")
    for idx, row in long_calls.iterrows():
        print(f"  Session: {row['Contact Session ID']}")
        print(f"    Type: {row['Front Desk Type']}")
        print(f"    Duration: {row['Duration Minutes']:.1f} minutes")
        print(f"    FD Start: {row['FD Start Time']}")
        print(f"    Call End: {row['Call End Time']}")

/var/folders/0k/z0xdft5548x02pfzbdd0x6w40000gn/T/ipykernel_82398/1726565419.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_main = pd.read_csv("combined_calls_CAR.csv")


Processing 17,893 front desk sessions...

FRONT DESK CALL DURATION STATISTICS
   Front Desk Type  Count  Average_Minutes  Median_Minutes  Min_Minutes  Max_Minutes  Std_Minutes Average_Label Median_Label
 FrontDeskTransfer   2545             5.01            4.37          0.0       467.30        10.09       5.0 min      4.4 min
FrontDeskTransfer1   7912             2.99            1.27          0.0       313.57         6.57       3.0 min      1.3 min
FrontDeskTransfer2   6895             4.10            2.92          0.0       171.47         5.62       4.1 min      2.9 min
FrontDeskTransfer3    541             2.05            0.27          0.0        63.52         5.13       2.0 min      0.3 min

Exported:
   - front_desk_duration_stats.csv (summary statistics)
   - front_desk_detailed_durations.csv (all session details)

OUTLIER CHECK:
Sessions > 30 minutes: 94
Sessions > 60 minutes: 20

Sample long calls (for verification):
  Session: 02a9424b-7e6c-44bd-9874-2f5e1e5e9eca
    Type: Fron